<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec04_pixel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec04_pixel.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第4回 画素処理
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，scikit-image，Matplotlib

MATLAB のライブスクリプト `code/matlab/vie_sec04_pixel.m` を Python に移植したもの。第4回のスライド（vie2026-04）の図と数値を計算する。画素処理（スカラー関数処理） $y=\phi(x)$ として，対比伸張・二値化閾値処理・ネガポジ変換・標準化と最小最大正規化・べき乗則変換，ヒストグラムとヒストグラム均等化，カラー画像処理（色調補正・擬似カラー）を順に扱う。記号は教科書に合わせ，度数を $h_x$，正規化度数を $p_x$，階調数を $L$，閾値を $\tau$ とする。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

画像処理には scikit-image，NumPy を使う。教科書の例題（「ネガポジ変換」「標準化と最小最大正規化」「ヒストグラム均等化」）の数値はスライドと一致することを `assert` で確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

写真は参考資料のサンプル画像 msipimg01〜08（512×512 のカラー）に限り，共通関数 `vie.msipimg` で読み込む。低対比・暗い・明るい画像など特定の性質が要る例は，これらを元にノートブックの中で作る。図の配色は，色そのものが内容の画像を除き，ロゴの 3 色（メインの緑，青，橙）と灰色にそろえる。

In [ ]:
import importlib
import os
import sys
import urllib.request

# Colab では毎回 GitHub から最新の vie.py を取得する（VieWork を clone した場合は同じフォルダの vie.py を使う）
if "google.colab" in sys.modules or not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv, hsv_to_rgb
from skimage.util import img_as_float, img_as_ubyte
from skimage.exposure import equalize_hist
from skimage.metrics import peak_signal_noise_ratio
import vie

vie = importlib.reload(vie)   # 取得し直した vie.py を読み込み直す

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

## 画素処理の数値例
画素処理は，すべての画素に同じ関数 $\phi(\cdot)$ を施す。例として 8-bit のネガポジ変換 $y=255-x$ を $2\times 3$ の配列に施す。

In [ ]:
x = np.array([[0, 64, 128], [200, 250, 255]])
y = 255 - x
print("x =\n", x)
print("y =\n", y)
assert np.array_equal(y, [[255, 191, 127], [55, 5, 0]])   # スライドの値

## 対比伸張と二値化閾値処理の関数
教科書の対比伸張（$x\in[0,1]$，$\gamma\in(0,1)$）

$$y=\phi(x)=\begin{cases} \frac{1-(1-2x)^\gamma}{2} & x<\frac12 \\ \frac{1+(2x-1)^\gamma}{2} & x\geq\frac12 \end{cases}$$

と，閾値 $\tau$ の二値化閾値処理を描く。$\gamma\to 0$ の対比伸張は $\tau=1/2$ の二値化閾値処理に一致する。

In [ ]:
def stretch(x, g):
    return (x < 0.5) * (1 - np.abs(1 - 2 * x)**g) / 2 + (x >= 0.5) * (1 + np.abs(2 * x - 1)**g) / 2


xx = np.linspace(0, 1, 1001)
fig, ax = plt.subplots(figsize=(3.2, 3.2), layout="constrained")
ax.plot(xx, xx, "--", color=cgray, lw=1.6)
ax.plot(xx, stretch(xx, 0.3), color=cmain, lw=2.4)
ax.grid(True)
ax.set(aspect="equal", xticks=[0, 0.5, 1], yticks=[0, 0.5, 1],
       xlabel="$x$", ylabel="$y=\\phi(x)$", title="$\\gamma=0.3$")
plt.show()

二値化閾値処理（$\tau=1/2$）

In [ ]:
fig, ax = plt.subplots(figsize=(3.2, 3.2), layout="constrained")
ax.plot(xx, xx, "--", color=cgray, lw=1.6)
ax.plot(xx, (xx >= 0.5).astype(float), color=cmain, lw=2.4)
ax.grid(True)
ax.set(aspect="equal", ylim=(-0.05, 1.05), xticks=[0, 0.5, 1], yticks=[0, 0.5, 1],
       xlabel="$x$", ylabel="$y=\\phi(x)$", title="$\\tau=0.5$")
plt.show()

## 対比伸張の例
石像の顔 msipimg05 をグレースケール（$256\times256$）にし，$0.3x+0.35$ で画素値を 0.35〜0.65 に押し込めた低対比画像を作る。これに対比伸張（$\gamma=0.3$）と二値化閾値処理（$\tau=0.5$）を施す。

In [ ]:
X = 0.3 * img_as_float(vie.msipimg(5, 256, "gray")) + 0.35   # 低対比画像を作る
print(f"range0 = [{X.min():.4f} {X.max():.4f}]")
Ycs = stretch(X, 0.3)
Yth = (X >= 0.5).astype(float)
print(f"range1 = [{Ycs.min():.4f} {Ycs.max():.4f}]")
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], X, "原画像")
show(axs[1], Ycs, "対比伸張")
show(axs[2], Yth, "二値化閾値処理")
plt.show()

## 基本的な輝度値変換
恒等変換，ネガポジ変換 $y=-x+1$，対数変換 $y=c\log(1+x)$（$c=1/\log 2$ で $[0,1]\to[0,1]$），べき乗則変換 $y=x^\gamma$ を比べる。

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.2), layout="constrained")   # 凡例は曲線に重ならないよう軸の右に置く
ax.plot(xx, xx, "--", color=cgray, lw=1.6, label="恒等")
ax.plot(xx, 1 - xx, color=ccool, lw=2.2, label="ネガポジ")
ax.plot(xx, np.log(1 + xx) / np.log(2), color=cwarm, lw=2.2, label="対数")
ax.plot(xx, xx**0.4, color=cmain, lw=2.2, label="べき乗則 $\\gamma=0.4$")
ax.plot(xx, xx**2.5, "-.", color=cmain, lw=2.2, label="べき乗則 $\\gamma=2.5$")
ax.grid(True)
ax.set(aspect="equal", xticks=[0, 0.5, 1], yticks=[0, 0.5, 1], xlabel="$x$", ylabel="$y=\\phi(x)$")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=9)
plt.show()

## ネガポジ変換
教科書の例題「ネガポジ変換」：0 が黒，1 が白の実数型画像 $\mathsf{x}\in[0,1]^{N_1\times N_2}$ には，$a=-1$ のスケール処理と $b=1$ のバイアス処理を施して

$$y=\phi(x)=-x+1,\quad x\in[0,1]$$

とする。整数型 $x\in\{0,1,\ldots,L-1\}$ では，画素値を $L-1$ で割って $[0,1]$ に写したものに同じ変換を施し，$L-1$ 倍して戻せば $y=L-1-x$ となる。石造りの建物 msipimg04（グレースケール，8-bit，$L=256$）で両者が一致することを確かめる。暗いアーチの奥の構造がネガ画像で見やすくなる。

In [ ]:
M = vie.msipimg(4, 256, "gray")
Mneg = 255 - M                                    # 整数型：y = L-1-x
errNeg = np.max(np.abs(img_as_float(Mneg) - (-img_as_float(M) + 1)))   # 実数型 y=-x+1 との差（0 になる）
print("errNeg =", errNeg)
assert errNeg < 1e-12
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6), layout="constrained")
show(axs[0], M, "原画像")
show(axs[1], Mneg, "ネガ画像")
plt.show()

8-bit（$L=256$）での数値例：

In [ ]:
L = 256
xn = np.array([0, 64, 200])
yn = (L - 1) - xn
print("，".join(f"{a}→{b}" for a, b in zip(xn, yn)))
assert np.array_equal(yn, [255, 191, 55])          # スライドの値

## 標準化と最小最大正規化
教科書の例題「標準化と最小最大正規化」。$4\times 6$ 実数配列 $\mathsf{x}$ の標本平均 $\mu$ と標本（不偏）分散 $\sigma^2$（$N-1$ で割る）を求め，

$$y=\frac{x-\mu}{\sigma},\qquad y=\frac{x-x_\mathrm{min}}{x_\mathrm{max}-x_\mathrm{min}}$$

を施す。

In [ ]:
xsd = np.array([[5, 5, 7, 6, 7, 2], [6, 2, 3, 2, 7, 2], [2, 2, 4, 4, 4, 6], [4, 4, 4, 4, 1, 2]], dtype=float)
Nsd = xsd.size                                    # 要素数（=24）
mu = xsd.mean()                                   # 標本平均
sigma2 = xsd.var(ddof=1)                          # 標本（不偏）分散
ystd = (xsd - mu) / np.sqrt(sigma2)               # 標準化
xmin, xmax = xsd.min(), xsd.max()
ymm = (xsd - xmin) / (xmax - xmin)                # 最小最大正規化
np.set_printoptions(precision=4, suppress=True)
print(f"N = {Nsd}，μ = {mu:.4f}，σ² = {sigma2:.4f}，x_min = {xmin:g}，x_max = {xmax:g}")
print("標準化 y =\n", ystd)
print("最小最大正規化 y =\n", ymm)

教科書の解答（小数点以下 4 桁）と照合する。

In [ ]:
ystdText = np.array([[0.5622, 0.5622, 1.6416, 1.1019, 1.6416, -1.0569],
                     [1.1019, -1.0569, -0.5172, -1.0569, 1.6416, -1.0569],
                     [-1.0569, -1.0569, 0.0225, 0.0225, 0.0225, 1.1019],
                     [0.0225, 0.0225, 0.0225, 0.0225, -1.5967, -1.0569]])
ymmText = np.array([[0.6667, 0.6667, 1.0000, 0.8333, 1.0000, 0.1667],
                    [0.8333, 0.1667, 0.3333, 0.1667, 1.0000, 0.1667],
                    [0.1667, 0.1667, 0.5000, 0.5000, 0.5000, 0.8333],
                    [0.5000, 0.5000, 0.5000, 0.5000, 0.0000, 0.1667]])
r4 = lambda v: np.round(v, 4)                       # 小数点以下 4 桁に丸める
assert abs(r4(mu) - 3.9583) < 1e-12 and abs(r4(sigma2) - 3.4330) < 1e-12
assert np.max(np.abs(r4(ystd) - ystdText)) < 1e-12
assert np.max(np.abs(r4(ymm) - ymmText)) < 1e-12
assert xmin == 1 and xmax == 7 and Nsd == 24
print("教科書の例題「標準化と最小最大正規化」の値と一致")

## べき乗則変換
実数 $x\in[0,1]$ では $y=x^\gamma$，整数 $x\in\{0,1,\ldots,L-1\}$ では $y=(L-1)\left(\frac{x}{L-1}\right)^\gamma$。$\gamma<1$ で明るく，$\gamma>1$ で暗くなる。

（スライドでは教科書の図 2.3 を使う。この曲線群の図は前回スライドの図に相当するもので，参考として残す。）

In [ ]:
gammas = [0.04, 0.1, 0.2, 0.4, 0.67, 1, 1.5, 2.5, 5, 10, 25]
colors = plt.cm.turbo(np.linspace(0, 1, len(gammas)))
fig, ax = plt.subplots(figsize=(5, 5), layout="constrained")
for g, c in zip(gammas, colors):
    ax.plot(xx, xx**g, color=c, lw=1.5, label=f"$\\gamma={g:g}$")
ax.grid(True)
ax.set(aspect="equal", xlabel="$x$", ylabel="$y=x^\\gamma$")
ax.legend(loc="lower right", ncol=2, fontsize=9)
plt.show()

数値例：$x=0.25$ のとき

In [ ]:
xp = 0.25
ypow = xp ** np.array([0.5, 2])            # γ=0.5 と γ=2
print("ypow =", ypow)
assert np.array_equal(ypow, [0.5, 0.0625])  # スライドの値

8-bit（$L=256$）で $x=64,\ \gamma=0.5$ のとき（四捨五入）：

In [ ]:
s = int(np.round(255 * (64 / 255)**0.5))
print("s =", s)
assert s == 128                              # スライドの値

## 暗い画像のべき乗則変換
石像の顔 msipimg05（グレースケール）を $x^3$ で暗くした画像を用意し，べき乗則変換（$\gamma=0.6,0.4,0.3$）で明るくする。

In [ ]:
S = img_as_float(vie.msipimg(5, 256, "gray"))**3   # 暗い画像を作る
meanS = S.mean()                                     # 平均画素値
print(f"meanS = {meanS:.2f}")
gd = [0.6, 0.4, 0.3]
fig, axs = plt.subplots(1, 4, figsize=(12, 3.4), layout="constrained")
show(axs[0], S, "原画像")
for k in range(3):
    show(axs[k + 1], S**gd[k], f"$\\gamma={gd[k]:g}$")
plt.show()

## 明るい画像のべき乗則変換
海岸 msipimg01（グレースケール）を $0.5+0.5x$ で明るく白っぽくした画像を用意し，べき乗則変換（$\gamma=3,4,5$）で暗くして濃淡を取り戻す。

In [ ]:
A = img_as_float(vie.msipimg(1, 256, "gray"))
B = 0.5 + 0.5 * A                          # 白っぽい（明るい）画像を作る
meanB = B.mean()
print(f"meanB = {meanB:.2f}")
gb = [3, 4, 5]
fig, axs = plt.subplots(1, 4, figsize=(12, 3.4), layout="constrained")
show(axs[0], B, "原画像")
for k in range(3):
    show(axs[k + 1], B**gb[k], f"$\\gamma={gb[k]:g}$")
plt.show()

## ヒストグラムの例
スイカ msipimg08（参考資料の図 2.5 と同じ画像．グレースケール，$256\times256$）から暗い・明るい・低対比・高対比の 4 種類の画像を作り，ヒストグラム（度数 $h_x$）を比べる。高対比画像は，次節のヒストグラム均等化（scikit-image の `equalize_hist`）で作る。

In [ ]:
def hist256(X):
    """8-bit 画像の度数 h_x（x=0,1,...,255）を数える．"""
    return np.bincount(X.ravel(), minlength=256)


def histplot(ax, h, color, xlab, ylab, hmax):
    """256 階調のヒストグラムを棒グラフで描く（縦軸の上限 hmax を共通にして比べやすくする）．"""
    ax.bar(np.arange(256), h, 1, color=color, edgecolor="none")
    ax.set(xlim=(0, 255), ylim=(0, hmax), xticks=[0, 128, 255], yticks=np.arange(0, hmax + 1, 2000),
           xlabel=xlab, ylabel=ylab)


C = vie.msipimg(8, 256, "gray")
Cf = img_as_float(C)
Cd = img_as_ubyte(np.clip(0.45 * Cf, 0, 1))          # 暗い画像
Cb = img_as_ubyte(np.clip(0.55 + 0.45 * Cf, 0, 1))   # 明るい画像
Cl = img_as_ubyte(np.clip(0.35 + 0.3 * Cf, 0, 1))    # 低対比画像
Ch = img_as_ubyte(equalize_hist(C))                  # 高対比画像
imgs = [Cd, Cb, Cl, Ch]
names = ["暗い", "明るい", "低対比", "高対比"]
hmaxAll = 6000                                       # 縦軸をそろえる（最大度数を含む）
print("最大度数:", max(hist256(v).max() for v in imgs))
fig, axs = plt.subplots(2, 4, figsize=(12, 5.6), layout="constrained")
for k in range(4):
    show(axs[0, k], imgs[k], names[k])
    histplot(axs[1, k], hist256(imgs[k]), cmain, "画素値 $x$", "度数 $h_x$", hmaxAll)
plt.show()

## ヒストグラム均等化の例
暗い画像と明るい画像にそれぞれヒストグラム均等化を施す。ヒストグラム均等化は，正規化度数の累積和（累積分布）$\sum_{v=0}^{x}p_v$ で画素値 $x$ を写す処理で，scikit-image の `equalize_hist` で計算できる（出力は $[0,1]$ の実数なので 8-bit に直す）。元の画像が違っても，結果のヒストグラムはどちらも一様に近く，似た画像になる。

In [ ]:
Ed, Eb = img_as_ubyte(equalize_hist(Cd)), img_as_ubyte(equalize_hist(Cb))
fig, axs = plt.subplots(2, 2, figsize=(7, 6), layout="constrained")
for k, (E, ttl) in enumerate(zip([Ed, Eb], ["暗い画像の均等化", "明るい画像の均等化"])):
    show(axs[0, k], E, ttl)
    histplot(axs[1, k], hist256(E), cmain, "画素値 $y$", "度数 $h_y$", hmaxAll)
plt.show()
psnrEq = peak_signal_noise_ratio(Ed, Eb)             # 均等化後の 2 枚はよく似ている
print(f"psnrEq = {psnrEq:.1f} dB")

## 例題：ヒストグラム均等化
教科書の例題「ヒストグラム均等化」。階調数 $L=8$ の $6\times 8$ 配列に，教科書の式

$$y=\phi(x)=\left\lfloor (L-1)\sum_{v=0}^{x}p_v \right\rceil,\quad x\in\{0,1,\ldots,L-1\}$$

でヒストグラム均等化を施す（$\lfloor\cdot\rceil$ は小数点第 1 位の四捨五入による整数化）。手順は，(1) 度数 $h_x$，(2) 正規化度数 $p_x=h_x/N$，(3) 参照表 $\phi$，(4) 出力配列 $\mathsf{y}=\phi(\mathsf{x})$ の順。

In [ ]:
xe = np.array([[2, 2, 3, 1, 2, 4, 1, 1],
               [3, 4, 4, 3, 2, 3, 4, 2],
               [4, 4, 4, 3, 4, 5, 5, 4],
               [0, 4, 6, 4, 2, 3, 4, 2],
               [2, 2, 2, 3, 5, 1, 3, 3],
               [2, 1, 4, 3, 4, 1, 6, 0]])
L = 8
N = xe.size                                       # 総画素数
hx = np.bincount(xe.ravel(), minlength=L)         # 度数 h_x（x=0,1,...,L-1）
px = hx / N                                       # 正規化度数 p_x
cum = (L - 1) * np.cumsum(px)                     # (L-1)Σ_{v=0}^{x} p_v
phi = np.round(cum).astype(int)                   # 参照表 φ(x)（四捨五入．ちょうど .5 の値は無い）
ye = phi[xe]                                      # 出力配列 y=φ(x)
hy = np.bincount(ye.ravel(), minlength=L)         # 均等化後の度数
print("N =", N)
print("h_x =", hx)
print("p_x =", px)
print("cum =", cum)
print("φ   =", phi)
print("y =\n", ye)
print("h_y =", hy)

教科書の度数分布表・画素値変換参照表・出力配列と照合する。

In [ ]:
hText = [2, 6, 11, 10, 14, 3, 2, 0]
phiText = [0, 1, 3, 4, 6, 7, 7, 7]
yText = np.array([[3, 3, 4, 1, 3, 6, 1, 1],
                  [4, 6, 6, 4, 3, 4, 6, 3],
                  [6, 6, 6, 4, 6, 7, 7, 6],
                  [0, 6, 7, 6, 3, 4, 6, 3],
                  [3, 3, 3, 4, 7, 1, 4, 4],
                  [3, 1, 6, 4, 6, 1, 7, 0]])
assert np.array_equal(hx, hText) and np.array_equal(phi, phiText) and np.array_equal(ye, yText)
assert abs(r4(cum[0]) - 0.2917) < 1e-12 and abs(r4(cum[1]) - 1.1667) < 1e-12 and abs(cum[-1] - (L - 1)) < 1e-12
assert np.array_equal(hy, [2, 6, 0, 11, 10, 0, 14, 5])   # スライドの均等化後の度数
print("教科書の例題「ヒストグラム均等化」の値と一致")

スライドの表に入れる正規化度数は分数で，度数が 0 のところは教科書と同じく 0 と書く。教科書の解答にある参照表の計算の途中式（$\phi(0)$，$\phi(1)$，$\phi(L-1)$）も示す。

In [ ]:
print("p_x:", "  ".join(f"{h}/{N}" if h else "0" for h in hx))
print(f"φ(0) = ⌊{L - 1}×{hx[0]}/{N}⌉ = ⌊{cum[0]:.4f}⌉ = {phi[0]}")
print(f"φ(1) = ⌊{L - 1}×({hx[0]}/{N}+{hx[1]}/{N})⌉ = ⌊{cum[1]:.4f}⌉ = {phi[1]}")
print(f"φ({L - 1}) = ⌊{L - 1}×{hx.sum()}/{N}⌉ = ⌊{cum[-1]:.4f}⌉ = {phi[-1]}")

均等化の前後の度数を棒グラフにする（前回演習課題（4）－2 の解説の図を参考に，縦軸をそろえる）。

In [ ]:
def histbar(ax, v, h, hmax, color, xlab, ylab):
    """度数の棒グラフを，棒の上に度数を添えて描く．"""
    ax.bar(v, h, 0.7, color=color, edgecolor="none")
    for vi, hi in zip(v, h):
        ax.text(vi, hi, str(hi), ha="center", va="bottom", fontsize=9)
    ax.set(xlim=(v[0] - 0.7, v[-1] + 0.7), ylim=(0, hmax), xticks=v, xlabel=xlab, ylabel=ylab)
    ax.spines[["top", "right"]].set_visible(False)


hmax = 15
fig, axs = plt.subplots(1, 2, figsize=(8, 3), layout="constrained")
histbar(axs[0], np.arange(L), hx, hmax, cmain, "画素値 $x$", "度数 $h_x$")
histbar(axs[1], np.arange(L), hy, hmax, cmain, "画素値 $y$", "度数 $h_y$")
plt.show()

## カラー画像処理：明るさの調整
マカロン msipimg03（$256\times256$）を明るくする。RGB 空間では R, G, B それぞれに $\gamma=0.4$ のべき乗則変換を施す。HSV 空間では明度 V だけに施し，色相 H と彩度 S は保つ（前回スライドの HSI 空間処理に相当）。HSV への変換には Matplotlib の `rgb_to_hsv`，`hsv_to_rgb` を使う。

In [ ]:
P = img_as_float(vie.msipimg(3, 256))
P = np.clip(P, 0, 1)                          # 縮小時のオーバーシュートを [0,1] に収める
Prgb = P**0.4
Phsv = rgb_to_hsv(P)
Phsv[:, :, 2] = Phsv[:, :, 2]**0.4
Phsv = hsv_to_rgb(Phsv)
sat = [rgb_to_hsv(v)[:, :, 1].mean() for v in (P, Prgb, Phsv)]   # 平均彩度（原画像，RGB 空間処理，HSV 空間処理）
print("sat =", np.round(sat, 2))
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], P, "原画像")
show(axs[1], Prgb, "RGB 空間処理")
show(axs[2], Phsv, "HSV 空間処理")
plt.show()

## カラー画像処理：彩度の調整
RGB 空間で R と G だけを 1.3 倍すると，色あい（色相）まで変わる。HSV 空間で彩度 S だけを 1.3 倍すれば，色あいは保たれる。

In [ ]:
Psat_rgb = P.copy()
Psat_rgb[:, :, :2] = np.minimum(1, 1.3 * P[:, :, :2])
Q = rgb_to_hsv(P)
Q[:, :, 1] = np.minimum(1, 1.3 * Q[:, :, 1])
Psat_hsv = hsv_to_rgb(Q)
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], P, "原画像")
show(axs[1], Psat_rgb, "RGB 空間処理")
show(axs[2], Psat_hsv, "HSV 空間処理")
plt.show()

## カラー画像処理：ヒストグラム均等化
R, G, B それぞれに均等化すると色あいが変わる。V（明度）だけに均等化すると色あいは保たれる。

In [ ]:
Peq_rgb = P.copy()
for c in range(3):
    Peq_rgb[:, :, c] = equalize_hist(P[:, :, c])
Q = rgb_to_hsv(P)
Q[:, :, 2] = equalize_hist(Q[:, :, 2])
Peq_hsv = hsv_to_rgb(Q)
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], P, "原画像")
show(axs[1], Peq_rgb, "R,G,B に適用")
show(axs[2], Peq_hsv, "V のみに適用")
plt.show()

## 擬似カラー
グレースケール画像の画素値 $x\in[0,1]$ を，位相をずらした三角波の参照表 $\vec{y}=(\phi_\mathrm{R}(x)\ \ \phi_\mathrm{G}(x)\ \ \phi_\mathrm{B}(x))^\top$ で色に対応づける（スカラー値を入力とするベクトル関数処理）。わずかな画素値の違いが色の違いとして強調される。曲線の色は R, G, B の各成分を表すので，ロゴの配色にはしない。

In [ ]:
def tri(x, ph):
    """周期 2/3 の三角波．"""
    return 1 - np.abs(2 * np.mod(1.5 * x + ph, 1) - 1)


fig, ax = plt.subplots(figsize=(5, 3.2), layout="constrained")   # 凡例は曲線の上の余白に横並びで置く
ax.plot(xx, tri(xx, 0), color=(0.85, 0, 0), lw=2.2, label="$\\phi_\\mathrm{R}(x)$")
ax.plot(xx, tri(xx, 1 / 3), color=(0, 0.6, 0), lw=2.2, label="$\\phi_\\mathrm{G}(x)$")
ax.plot(xx, tri(xx, 2 / 3), color=(0, 0, 0.85), lw=2.2, label="$\\phi_\\mathrm{B}(x)$")
ax.grid(True)
ax.set(xlabel="画素値 $x$", ylabel="出力", ylim=(0, 1.42), yticks=[0, 0.5, 1], xticks=[0, 0.5, 1])
ax.legend(loc="upper center", ncol=3, fontsize=9)
plt.show()

In [ ]:
G = img_as_float(vie.msipimg(8, 256, "gray"))   # スイカ（グレースケール）
Yps = np.stack([tri(G, 0), tri(G, 1 / 3), tri(G, 2 / 3)], axis=2)
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6), layout="constrained")
show(axs[0], G, "モノクロ画像")
show(axs[1], Yps, "擬似カラー表示画像")
plt.show()

## 例題：擬似カラー
前回スライドの例題。$4\times4$ 配列 $x\in\{0,1,2,3\}$ を変換テーブル（R, G, B 各 2 bit）で擬似カラー表示する。変換テーブルの行が画素値 $x=0,1,2,3$ に対応する。

In [ ]:
xq = np.array([[0, 2, 2, 3], [1, 1, 1, 2], [1, 1, 1, 1], [1, 1, 1, 0]])
lut = np.array([[3, 0, 0], [0, 3, 0], [0, 0, 3], [3, 0, 3]])   # 行 x=0,1,2,3 の (R,G,B)
lutName = ["赤", "緑", "青", "マゼンタ"]                       # 各行の色の名前
Yq = lut[xq] / 3                                               # 配列 y（RGB，[0,1] に正規化）
print("lut =\n", lut)

配列 $\mathsf{x}$ は，画素値を灰色の濃淡（0 が黒に近く，3 が白に近い）で表し，値を重ねて描く。配列 $\mathsf{y}$ は変換後の色で描く。

In [ ]:
def cellplot(ax, rgb, labels=None):
    """配列を升目として描く（labels を与えれば値を重ねる）．"""
    nr, nc = rgb.shape[:2]
    ax.imshow(rgb, interpolation="nearest")
    for k in np.arange(nc + 1) - 0.5:
        ax.plot([k, k], [-0.5, nr - 0.5], "k", lw=1.5)
    for k in np.arange(nr + 1) - 0.5:
        ax.plot([-0.5, nc - 0.5], [k, k], "k", lw=1.5)
    if labels is not None:
        for i in range(nr):
            for j in range(nc):
                txt = "w" if rgb[i, j].mean() < 0.5 else "k"   # 暗い升目は白字
                ax.text(j, i, str(labels[i, j]), ha="center", va="center", fontsize=20, color=txt)
    ax.set_axis_off()


fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")
cellplot(axs[0], np.repeat((0.1 + 0.8 * xq / 3)[:, :, None], 3, axis=2), xq)
axs[0].set_title("配列 $\\mathsf{x}$")
cellplot(axs[1], Yq)
axs[1].set_title("配列 $\\mathsf{y}$")
plt.show()

変換テーブルと，各画素値の色の対応を示す。色は教科書の記法どおり列ベクトル $\vec{y}=\boldsymbol{\phi}(x)=(y_\mathrm{R}\ y_\mathrm{G}\ y_\mathrm{B})^\top$ で書く。

In [ ]:
print("x :", " ".join(str(k) for k in range(lut.shape[0])))
for c, ch in enumerate("RGB"):
    print(f"{ch} :", " ".join(str(v) for v in lut[:, c]))
for k in range(lut.shape[0]):
    print(f"φ({k}) = ({lut[k, 0]} {lut[k, 1]} {lut[k, 2]})^T（{lutName[k]}）")

## まとめ
- 画素処理は全画素に同じスカラー関数 $y=\phi(x)$ を施す
- 標準化・最小最大正規化はスケール処理とバイアス処理の組み合わせ
- べき乗則変換は $\gamma<1$ で明るく，$\gamma>1$ で暗くする
- ヒストグラム均等化は正規化度数の累積和から参照表を自動で作る
- カラー画像は HSV（HSI）空間で明度・彩度を操作すると色あいを保てる

© Copyright, Shogo MURAMATSU, All rights reserved.